### Introduction

This codelab is based on https://www.rainbowbreeze.it/serving-gemma-2b-for-free-using-google-colab/ blogpost.

Author: Alfredo Morresi

## Install Ollama inside this Colab

In [ ]:
!sudo apt-get install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

### Download packages to create the Cloudflare tunnel

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod a+x cloudflared

### Prepare the thread to expose the Ollama via cloudflare

In [ ]:
import os
import subprocess
import threading
import time
import socket

# 1. Set Environment Variables
os.environ.update({'OLLAMA_HOST': '0.0.0.0:11434'})

# 2. Function to start the Ollama Server
def run_ollama_server():
    print("Starting Ollama server...")
    # Make sure ollama is installed: !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.run(["ollama", "serve"])

# 3. Your existing Tunnel Thread
def iframe_thread(port):
    print("Watcher Thread: Waiting for Ollama to bind to port...")
    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()

    print("Ollama detected! Starting Cloudflare tunnel...")
    # Ensure cloudflared is in the current directory or PATH
    p = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                         stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("\n" + "="*30)
            print("OLLAMA SERVER PUBLIC URL:")
            print(l[l.find("http"):].strip())
            print("="*30 + "\n")

# --- EXECUTION ---

# Start the Ollama Server in the background
threading.Thread(target=run_ollama_server, daemon=True).start()

# Start the Cloudflare Watcher
threading.Thread(target=iframe_thread, daemon=True, args=(11434,)).start()

# Keep the main thread alive so the background threads don't die
while True:
    time.sleep(10)

### Start serving Ollama

In [ ]:

import os
import threading
import subprocess
import time

def ollama():
    # Set environment variables
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    os.environ['OLLAMA_ORIGINS'] = '*'

    # Use full path or ensure it's in system PATH
    # 'serve' starts the daemon
    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    # Keep the thread alive so you can see logs if it fails
    for line in process.stderr:
        print(f"[Ollama]: {line.strip()}")

# Start the thread
ollama_thread = threading.Thread(target=ollama, daemon=True)
ollama_thread.start()

# Give it a few seconds to initialize before running other cells
time.sleep(5)
print("Ollama server should be running.")

### Use Ollama API from the command line

Finally, it's time to call the exposed Ollama API, with ```<your_cloudflare_url>``` as something like https://auditor-protect-vids-slots.trycloudflare.com.


Install the gemma:2b model
```
curl <your_cloudflare_url>/api/pull -d '{ "name": "gemma:2b" }'
```

Cache the model, to save some time between calls (by default, after 5 mins the model is discarded from memory)
```
curl <your_cloudflare_url>/api/generate -d '{"model": "gemma:2b", "keep_alive": -1}'
```

Finally, ask for the prompt
```
curl <your_cloudflare_url>/api/generate --max-time 600 -d '{"model": "gemma:2b", "stream":false, "prompt": "generate a bash script to ask the user for a url, for a message, and then call that url using the message as json payload"}' | jq ".response"
```